In [1]:
import warnings
from config import *
import os
import re
import pandas as pd
import numpy as np

# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Do not truncate column values
pd.set_option('display.max_colwidth', None)

In [16]:
randhrs = "data/original data/randhrs1992_2020v2.dta"
hrs_tracker = "data/original data/trk2022tr_r.dta"

In [17]:
# Set wave e.g:
# year = 2 * (wave - 5) from 2002 to 2016

wave = 13
year = 2 * (wave - 5)

In [18]:
# Time-invariant variables (from base interview)
timeinvariant_household_col = ['hidpn']
timeinvariant_respondent_col = ['gender', 'edyrs', 'bplace']
timeinvariant_col = [f'h{col}' for col in timeinvariant_household_col] + \
                    [f'ra{col}' for col in timeinvariant_respondent_col]

# Wave-specific variables for respondent and household
wave_household_col = ['child'] # Number of children

wave_respondent_col = [
    'lbrf',     # Labor force status
    'shlt',     # Self-rated health
    'agey_m',   #age
    'height',
    'weight',
    'smokev',   # Smoker status
    'proxy',    # Proxy interview indicator
    'imrc',  # immediate recall of a list of 10 words
    'dlrc',  # delayed recall of a list of 10 words
    'ser7',  # five trials of serial 7s
    'bwc20',  # backward counting
    'prmem',  # proxy rating of respondent memory
    'iadl5a',  # iadl5a = sum(phonea, moneya, medsa, shopa, mealsa), referred as "iadlza" in appendix
    'cendiv', # Census Division
    'mstat', # Marital Status
    'effort', # CESD Everything an effort
    'hibpe', # Ever had high blood pressure
    'diabe', # Ever had diabetes
    'cancre', # Ever had cancer
    'lunge', # Ever had lung disease
    'hearte', # Ever had heart problems
    'stroke', # Ever had stroke
    'arthre', # Ever had arthritis
    'slfmem', # Self-rated memory
    'livpar', # Number of living parents
    'momage', # Mother age current/at death
    'dadage', # Father age current/at death
    'livsib', # Number of living siblings
    'hlthlm', # Health problems limit work
    'hosp',	# Hospital stay(Overnight hospital stay since PrvIvw before death)
    'nrshom', # Nursing home stay (whether the Respondent reports any overnight nursing home stay in the reference period)
    'nrstim', # Nursing home stays
    'nrsnit', # Nights in nursing home (number of nights over all stays)
    'doctim', # Doctor visits (reported number of visits)
    'depres', # CESD Felt depressed
    'sleepr', # CESD Sleep was restless
    'whappy', # CESD Was happy
    'flone',    # CESD Felt lonely
    'fsad',     # CESD Felt sad
    'going',    # CESD Could not get going
    'enlife',   # CESD Enjoyed life
    'drink',    # Ever drinks any alcohol
    'smoken',   # Smokes now
    'toilta',   # Some Difficulty-Using the toilet
    'adl5a',    # Some Difficulty-IADLs /0-3
    'mapa',     # Some Difficulty-Use a map
    'walksa',   # Some Difficulty-Walk sev blocks
    'walk1a',   # Some Difficulty-Walk one block
    'sita',     # Some Difficulty-Sit for 2 hours
    'chaira',   # Some Difficulty-Get up fr chair
    'climsa',   # Some Difficulty-Clmb sev flt str
    'clim1a',   # Some Difficulty-Clmb 1 flt stair
    'stoopa',   # Some Difficulty-Stoop/Kneel/Crch
    'lifta',    # Some Difficulty-Lift/carry 10 lbs
    'dimea',    # Some Difficulty-Pick up a dime
    'armsa',    # Some Difficulty-Rch/xtnd arms up
    'pusha',    # Some Difficulty-Push/pull lg obj
    'mobila',   # Some Difficulty-Mobility /0-5
    'lgmusa',   # Some Diff-Large Muscle /0-4
    'grossa',   # Walk1/R,Clim1,Bed,Bath/0-5
    'finea',    # Dime/Eat/Dress /0-3
]

if wave==6:
    wave_respondent_col.append('vigact') # (For w6): Whether vigorous phys act 3+/wk
else:
    wave_respondent_col.append('vgactx') # (For w7-w15) Whether vigorous phys act 3+/wk, Freq vigorous phys active {finer scale}

# Construct final columns to load
columns = timeinvariant_col + \
          [f'h{wave}{col}' for col in wave_household_col] + \
          [f'r{wave}{col}' for col in wave_respondent_col]

# Read the selected columns from the dataset
chunks_main = pd.read_stata(randhrs, columns=columns)

# Map the correct 'prfin' column based on year
col_prfin = {2:'HA011',
             4:'JA011',
             6:'KA011',
             8:'LA011',
             10:'MA011',
             12: 'NA011',
             14: 'OA011',
             16: 'PA011',
             18: 'QA011',
             20: 'RA011',
             }
original_prfin_col = col_prfin[year]
new_prfin_col = f'r{wave}prfin'

# Load prefin column dataset 
prfin_data = pd.read_csv('data/original data/'+f'h{year}A_R.csv')

In [19]:
# Create unique person ID
#prfin_data['hhidpn'] = (prfin_data['HHID'] + prfin_data['PN']).astype(int)
prfin_data['hhidpn']=(prfin_data['HHID'].astype(str)+'0'+prfin_data['PN'].astype(str)).astype(int)

# Recode original prfin values into categories
prfin_data[new_prfin_col] = pd.cut(
    prfin_data[original_prfin_col],
    bins=[0, 1, 2, 3],
    labels=['1.none', '2.some', '3.prevented'],
    ordered=True
)

# Keep only hhidpn and recoded prfin variable
prfin_data = prfin_data[['hhidpn', new_prfin_col]]

# Merge with main dataset on hhidpn
chunks = pd.merge(
    chunks_main,
    prfin_data,
    on='hhidpn',
    how='left'
)

In [20]:
# Define new column name for race/ethnicity based on wave
raceeth_col = f'r{wave}raceeth'

# Load relevant columns from HRS tracker file
raceeth_tracker = pd.read_stata(
    hrs_tracker,
    columns=['HHID', 'PN', 'RACE', 'HISPANIC', 'NIWWAVE']
)

# Generate unique person ID
raceeth_tracker['hhidpn'] = (raceeth_tracker['HHID'] + raceeth_tracker['PN']).astype(int)

# Recode race/ethnicity following HRS coding logic:
# 0 = Non-Hispanic White
# 1 = Non-Hispanic Black
# 2 = Hispanic
# 3 = Non-Hispanic Other
raceeth_tracker.loc[raceeth_tracker['HISPANIC'].isin([1, 2, 3]), raceeth_col] = 2
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'].isin([0, 5])) & (raceeth_tracker['RACE'] == 1), raceeth_col] = 0
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'].isin([0, 5])) & (raceeth_tracker['RACE'] == 2), raceeth_col] = 1
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'].isin([0, 5])) & (raceeth_tracker['RACE'] == 7), raceeth_col] = 3

# Step 2: Filter for rows where NIWWAVE == 1
raceeth_tracker = raceeth_tracker[raceeth_tracker['NIWWAVE'] == 1]

# Drop original identifiers and raw race/ethnicity variables
raceeth_tracker.drop(columns=[ 'HISPANIC', 'RACE'], inplace=True)

# Check distribution of the recoded race/ethnicity variable
raceeth_tracker[raceeth_col].value_counts(dropna=False)


r13raceeth
0.0    14098
1.0     4146
2.0     2911
3.0      722
NaN       16
Name: count, dtype: int64

In [21]:
chunks.describe(include='all')

,hhidpn,ragender,raedyrs,rabplace,h13child,r13lbrf,r13shlt,r13agey_m,r13height,r13weight,r13smokev,r13proxy,r13imrc,r13dlrc,r13ser7,r13bwc20,r13prmem,r13iadl5a,r13cendiv,r13mstat,r13effort,r13hibpe,r13diabe,r13cancre,r13lunge,r13hearte,r13stroke,r13arthre,r13slfmem,r13livpar,r13momage,r13dadage,r13livsib,r13hlthlm,r13hosp,r13nrshom,r13nrstim,r13nrsnit,r13doctim,r13depres,r13sleepr,r13whappy,r13flone,r13fsad,r13going,r13enlife,r13drink,r13smoken,r13toilta,r13adl5a,r13mapa,r13walksa,r13walk1a,r13sita,r13chaira,r13climsa,r13clim1a,r13stoopa,r13lifta,r13dimea,r13armsa,r13pusha,r13mobila,r13lgmusa,r13grossa,r13finea,r13vgactx,r13prfin
count,4.240500e+04,42405,42272,42366,20494.000000,20912,20888,20912.000000,20795.000000,20686.000000,20824,20912,19971.000000,19971.000000,19971.000000,19971,935,20874.000000,20893,20880,19933,20912,20912,20912,20912,20912,20912,20912,19971,20632.000000,20508.000000,19979.000000,20877.000000,20243,20737,20777,20723.000000,20708.000000,19048.000000,19947,19927,19902,19944,19937,19889,19927,20897,20818,20851,20876.000000,19596,20739,20788,20760,20803,20167,20586,20697,20606,20822,20817,20465,20872.000000,20873.000000,20876.000000,20875.000000,20808,941
unique,NaN,2,18,11,NaN,7,5,NaN,NaN,NaN,2,2,NaN,NaN,NaN,3,5,NaN,10,7,2,2,2,2,2,2,2,2,5,NaN,NaN,NaN,NaN,2,2,2,NaN,NaN,NaN,2,2,2,2,2,2,2,2,2,2,NaN,2,2,2,2,2,2,2,2,2,2,2,2,NaN,NaN,NaN,NaN,5,3
top,NaN,2.female,12.0,3.en central,NaN,5.retired,3.good,NaN,NaN,NaN,1.yes,0.not proxy,NaN,NaN,NaN,"2.correct, 1st try",5.poor,NaN,5.s atlantic,1.married,0.no,1.yes,0.no,0.no,0.no,0.no,0.no,1.yes,3.good,NaN,NaN,NaN,NaN,0.no,0.no,0.no,NaN,NaN,NaN,0.no,0.no,1.yes,0.no,0.no,0.no,1.yes,1.yes,0.no,0.no,NaN,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,NaN,NaN,NaN,NaN,5.never,3.prevented
freq,NaN,23763,13066,6711,NaN,9926,6993,NaN,NaN,NaN,11298,19971,NaN,NaN,NaN,18551,325,NaN,5251,11378,14369,12498,15439,17953,18740,16117,19075,11447,8125,NaN,NaN,NaN,NaN,12618,15523,19919,NaN,NaN,NaN,17258,13582,17063,16466,15913,16004,18066,12135,17765,19474,NaN,16811,14219,17397,16262,12856,10954,16428,11280,15202,19233,16965,14757,NaN,NaN,NaN,NaN,11462,489
mean,2.896891e+08,NaN,NaN,NaN,3.016444,NaN,NaN,65.702085,1.679192,81.860409,NaN,NaN,5.357669,4.314756,3.438786,NaN,NaN,0.363802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.411012,75.991223,72.329045,3.282033,NaN,NaN,NaN,0.088549,9.240728,9.889227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.400604,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.196340,1.320845,0.606103,0.232240,NaN,NaN
std,2.519221e+08,NaN,NaN,NaN,2.023787,NaN,NaN,11.794857,0.105413,20.111794,NaN,NaN,1.678671,1.980367,1.680962,NaN,NaN,0.973842,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.656530,13.972513,14.261619,2.624733,NaN,NaN,NaN,1.754893,83.366471,20.971274,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.021183,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.559719,1.398788,1.210849,0.586656,NaN,NaN
min,1.010000e+03,NaN,NaN,NaN,0.000000,NaN,NaN,21.000000,0.914400,27.215400,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,17.000000,12.000000,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN
25%,7.333204e+07,NaN,NaN,NaN,2.000000,NaN,NaN,56.000000,1.600200,68.038500,NaN,NaN,4.000000,3.000000,2.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,70.000000,64.000000,1.000000,NaN,NaN,NaN,0.000000,0.000000,2.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN
50%,2.046920e+08,NaN,NaN,NaN,3.000000,NaN,NaN,64.000000,1.676400,79.378250,NaN,NaN,5.000000,4.000000,4.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,78.000000,75.000000,3.000000,NaN,NaN,NaN,0.000000,0.000000,5.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,N

In [7]:
# Merge race/ethnicity tracker data into the main dataset using `hhidpn` as the key
chunks = pd.merge(
    chunks,                # main analysis dataset (already includes wave, prfin, etc.)
    raceeth_tracker,       # race/ethnicity tracker data
    on='hhidpn',           # merge key
    how='left'             # keep all cases from main dataset
)
chunks.head()

,hhidpn,ragender,raedyrs,rabplace,h13child,r13lbrf,r13shlt,r13agey_m,r13height,r13weight,r13smokev,r13proxy,r13imrc,r13dlrc,r13ser7,r13bwc20,r13prmem,r13iadl5a,r13cendiv,r13mstat,r13effort,r13hibpe,r13diabe,r13cancre,r13lunge,r13hearte,r13stroke,r13arthre,r13slfmem,r13livpar,r13momage,r13dadage,r13livsib,r13hlthlm,r13hosp,r13nrshom,r13nrstim,r13nrsnit,r13doctim,r13depres,r13sleepr,r13whappy,r13flone,r13fsad,r13going,r13enlife,r13drink,r13smoken,r13toilta,r13adl5a,r13mapa,r13walksa,r13walk1a,r13sita,r13chaira,r13climsa,r13clim1a,r13stoopa,r13lifta,r13dimea,r13armsa,r13pusha,r13mobila,r13lgmusa,r13grossa,r13finea,r13vgactx,r13prfin,HHID,PN,NIWWAVE,r13raceeth
0,1010,1.male,16.0,2.mid atlantic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010,2.female,8.0,3.en central,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3010,1.male,12.0,9.pacific,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,000003,010,1.0,0.0
3,3020,2.female,16.0,8.mountain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,000003,020,1.0,0.0
4,10001010,1.male,12.0,2.mid atlantic,0.0,5.retired,2.very good,76.0,1.8034,63.5026,0.no,0.not proxy,5.0,5.0,4.0,"2.correct, 1st try",NaN,0.0,2.mid atlantic,8.never married,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,3.good,0.0,79.0,66.0,1.0,0.no,1.yes,0.no,0.0,0.0,8.0,0.no,0.no,1.yes,0.no,0.no,0.no,1.yes,0.no,0.no,0.no,0.0,NaN,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,0.no,NaN,0.0,0.0,0.0,0.0,2.>1 per week,NaN,010001,010,1.0,0.0


In [8]:
# Convert time-invariant categorical variables to float (delete float setting)
timeinvariant_categorical_cols = ['ragender', 'raedyrs']

for col in timeinvariant_categorical_cols:
    chunks[col] = chunks[col].cat.codes.astype(float)
    chunks.loc[chunks[col] == -1, col] = float('nan')

# Recode one-hot indicator
for col in [
    'smokev', 'proxy', 'prmem', 'prfin','bwc20',  'effort',
    'hibpe', 'diabe', 'cancre', 'lunge', 'hearte', 'stroke',
#    'vigact', #only for wave 6
    'vgactx',
    'arthre', 'slfmem', 'hlthlm', 'hosp', 'nrshom',
    'depres','sleepr', 'whappy', 'flone', 'fsad', 'going',
    'enlife', 'drink', 'smoken', 'toilta', 'mapa', 'walksa',
    'walk1a', 'sita', 'chaira', 'climsa', 'clim1a', 'stoopa',
    'lifta', 'dimea', 'armsa','pusha'
]:

# delete categorical variables: 'cendiv', 'mstat', 'bplace' due to non-ordinal

    wave_col = f'r{wave}{col}'
    chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
    chunks.loc[chunks[wave_col] == -1, wave_col] = float('nan')

# Bin education into stage groups
raedstg: ['1.0-7', '2.8-11', '3.12', '4.13+']
chunks['raedstg'] = pd.cut(
    chunks['raedyrs'],
    bins=[-1, 7, 11, 12, chunks['raedyrs'].max()],
    labels=['1.0-7', '2.8-11', '3.12', '4.13+']
)

# Convert all wave-specific categorical variables to float
wave_categorical_cols = [
    'lbrf',     # Labor force status
    'shlt',     # Self-rated health
]

for col in wave_categorical_cols:
    wave_col = f'r{wave}{col}'
    chunks[wave_col] = chunks[wave_col].cat.codes.astype(float)
    chunks.loc[chunks[wave_col] == -1, wave_col] = float('nan')

C:\Users\13022\AppData\Local\Temp\ipykernel_8660\914134105.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1. -1. -1. ...  1. -1.  1.]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
C:\Users\13022\AppData\Local\Temp\ipykernel_8660\914134105.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1. -1. -1. ...  0. -1.  0.]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
C:\Users\13022\AppData\Local\Temp\ipykernel_8660\914134105.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1. -1. -1. ... -1. -1. -1.]' has dtype incompatible with category,

In [9]:
# Drop the original 'raedyrs' column (already binned into 'raedstg' and encoded into 'redleq17')
chunks.drop(columns=['raedyrs'], inplace=True)

# (Optional) Reorder columns: Move key variables like hhidpn and race/ethnicity to the front
cols_order = ['hhidpn']
if f'r{wave}raceeth' in chunks.columns:
    cols_order.append(f'r{wave}raceeth')
if 'ragender' in chunks.columns:
    cols_order.append('ragender')
if 'raedstg' in chunks.columns:
    cols_order.append('raedstg')
if 'redleq17' in chunks.columns:
    cols_order.append('redleq17')

# Add remaining columns after key ones
cols_remaining = [col for col in chunks.columns if col not in cols_order]
chunks = chunks[cols_order + cols_remaining]
chunks[f'r{wave}raceeth'].value_counts(dropna=False)

r13raceeth
NaN    20528
0.0    14098
1.0     4146
2.0     2911
3.0      722
Name: count, dtype: int64

In [10]:
# Define variable names
proxy_col = f'r{wave}proxy'
demscr_col = f'r{wave}demscr'
demcls_col = f'r{wave}demcls'

# Define column groups used to calculate cognitive score
self_dem_cols_wave = [f'r{wave}{col}' for col in SELF_DEM_COLS]
proxy_dem_cols_wave = [f'r{wave}{col}' for col in PROXY_DEM_COLS]

# Recode for self-respondents
self_mask = (chunks[proxy_col] == 0)
print(self_mask.sum())
chunks.loc[self_mask, demscr_col] = chunks.loc[self_mask, self_dem_cols_wave].sum(
    axis=1, min_count=len(self_dem_cols_wave)
)
chunks.loc[self_mask & (chunks[demscr_col] <= 6), demcls_col] = 1.0
chunks.loc[self_mask & (chunks[demscr_col] > 6), demcls_col] = 0.0

# Recode for proxy-respondents
proxy_mask = (chunks[proxy_col] == 1)
chunks.loc[proxy_mask, demscr_col] = chunks.loc[proxy_mask, proxy_dem_cols_wave].sum(
    axis=1, min_count=len(proxy_dem_cols_wave)
)
chunks.loc[proxy_mask & (chunks[demscr_col] >= 6), demcls_col] = 1.0
chunks.loc[proxy_mask & (chunks[demscr_col] < 6), demcls_col] = 0.0

# Drop rows with missing demscr (as they can't be classified)
# chunks.dropna(subset=[demscr_col], inplace=True)

# Drop raw demscr score now that demcls has been derived
chunks.drop(columns=[demscr_col], inplace=True)

19971


In [11]:
def remove_wave_prefix(col, wave):
    # Remove 'ra', 'r11', 'ha', 'h11' prefixes only
    return re.sub(rf'^(r|h)(a|{wave})', '', col)

# Apply clean renaming to all columns
chunks.rename(
    columns={col: remove_wave_prefix(col, wave) for col in chunks.columns},
    inplace=True
)

In [12]:
core_feature_cols = [
    'HHID', 'PN', 'hhidpn', 'NIWWAVE', # keys to merge, not x variables
    # all seleted x variables
    'edstg', "child", "lbrf", "shlt", "agey_m", "height", "weight", "smokev", "proxy", "cendiv", "mstat", "effort", "hibpe", "diabe",
    'vgactx',
    #'vigact', # for wave 6 only
    "slfmem", "livpar", "momage", "dadage", "livsib", "hlthlm", "hosp", "nrshom", "nrstim", "nrsnit", "doctim", "depres", "sleepr", "whappy", "flone", "fsad", "going", "enlife", "drink", "smoken", "cancre", "lunge", "hearte", "stroke", "arthre", "toilta",
    "adl5a", "mapa", "walksa", "walk1a", "sita", "chaira", "climsa", "clim1a", "stoopa", "lifta", "dimea", "armsa", "pusha",
    "mobila", "lgmusa", "grossa", "finea", "raceeth","gender",
    "demcls" # y: Binary cognitive impairment classification
]

chunks_new=chunks[core_feature_cols]
# optional: dropping all rows with age ≤ 50, and demcls == 1, exclude all people who has dementia and age less than or equal to 50 on baseline year (2000,2006,2008,2010) to predict
#chunks_new = chunks_new[chunks_new['agey_m'] > 50].copy().reset_index(drop=True)
#chunks_new = chunks_new[chunks_new['demcls'] != 1].copy().reset_index(drop=True)

# Convert the 'raceeth' column to a categorical variable
chunks_new['raceeth'] = chunks_new['raceeth'].astype('category')

C:\Users\13022\AppData\Local\Temp\ipykernel_8660\717615465.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chunks_new['raceeth'] = chunks_new['raceeth'].astype('category')


In [13]:
if year<10:
    year=f'0{year}'

In [14]:
chunks_new = pd.get_dummies(chunks_new,columns=chunks_new.select_dtypes(include=['category']).columns.tolist(),drop_first=True)
chunks_new.to_csv('data/preprocessed data/'+f'20{year}.csv',na_rep="NA" ,index=False)

In [15]:
#Select 'demcls' and rename it to '__demcls' and HHID,PN, hhidpn
chunks_new.rename(columns={'demcls':f'{year}demcls'}, inplace=True)
y = chunks_new[['HHID','PN', 'hhidpn' , f'{year}demcls']]
y.to_csv('data/preprocessed data/'+f'_{year}y.csv',na_rep="NA" ,index=False)